In [1]:
import pandas as pd
import numpy as np
import scipy.stats as st
import statsmodels.formula.api as smf
import statsmodels.stats as sms
import statsmodels.api as sm

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

In [2]:
base = pio.templates["simple_white"]
custom_template = base.layout.template

custom_template.layout.update(
    margin=dict(l=50, r=0, t=20, b=50),
    xaxis=dict(
        title_standoff=0,
        ticks="outside",
        showgrid=True,
        gridcolor='#bbb'
    ),
    yaxis=dict(
        title_standoff=0,
        ticks="outside",
        showgrid=True,
        gridcolor='#bbb'
    ),
    colorscale=dict(
        sequential=px.colors.sequential.Inferno,
        diverging=px.colors.diverging.curl_r
    ),
    colorway=px.colors.qualitative.T10
)

pio.templates["tight"] = custom_template
pio.templates.default = "tight"

In [3]:
kc = pd.read_csv('kc_house_data.csv')

In [4]:
kc['date'] = pd.to_datetime(kc.date)

sqft_to_sqmeters = 0.09290303997

kc = kc.astype({'sqft_living': float, 'sqft_lot': float, 
                'sqft_above': float, 'sqft_basement': float,
                'sqft_living15': float, 'sqft_lot15': float,
                'yr_renovated': float, 'yr_built': float,
                'bedrooms': float, 'waterfront': float,
                'view': float, 'condition': float, 'grade': float})

kc.loc[:,['sqft_living', 'sqft_lot', 'sqft_above', 'sqft_basement', 'sqft_living15', 'sqft_lot15']] =\
kc.loc[:,['sqft_living', 'sqft_lot', 'sqft_above', 'sqft_basement', 'sqft_living15', 'sqft_lot15']] * sqft_to_sqmeters

kc.columns = ['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqm_living',
       'sqm_lot', 'floors', 'waterfront', 'view', 'condition', 'grade',
       'sqm_above', 'sqm_basement', 'yr_built', 'yr_renovated', 'zipcode',
       'lat', 'long', 'sqm_living15', 'sqm_lot15']

kc = kc.loc[:, ['id', 'date', 'price', 
            'floors', 'bedrooms', 'bathrooms', 
            'yr_built', 'yr_renovated',
            'waterfront', 'view', 'condition', 'grade', 
            'sqm_living', 'sqm_lot', 
            'sqm_above', 'sqm_basement',
            'sqm_living15', 'sqm_lot15',
            'lat', 'long']]

kc.loc[kc.sqm_basement == 0, 'sqm_basement'] = np.nan
kc.loc[kc.yr_renovated == 0, 'yr_renovated'] = np.nan

In [5]:
kc_log = kc.copy()
kc_log.loc[:, ['price', 'sqm_living', 'sqm_lot', 'sqm_above', 'sqm_basement', 'sqm_living15', 'sqm_lot15']] = \
np.log10(kc_log.loc[:, ['price', 'sqm_living', 'sqm_lot', 'sqm_above', 'sqm_basement', 'sqm_living15', 'sqm_lot15']])

In [6]:
kc_log.yr_renovated = kc_log.yr_renovated.fillna(kc_log.yr_built)
kc_log.loc[:,'sqm_basement'] = kc_log.loc[:,'sqm_basement'].fillna(0)
kc.yr_renovated = kc.yr_renovated.fillna(kc.yr_built)
kc.loc[:,'sqm_basement'] = kc.loc[:,'sqm_basement'].fillna(0)

In [7]:
kc_z=kc_log.copy()
for col in kc_z.drop(columns=['id', 'date']).columns:
    kc_z.loc[:,col] = st.zscore(kc_z.loc[:,col])

In [ ]:
X = kc.drop(columns=['id', 'date', 'price', 'yr_renovated',
                                'sqm_above', 'sqm_basement',
                                'sqm_living15', 'sqm_lot15',
                                'lat', 'long'])
# X = kc_log.grade.values.reshape(-1, 1)
y = kc.price
reg = linear_model.LinearRegression().fit(X, y)
reg.score(X, y)
reg.coef_.reshape(-1,1)

In [24]:
# X = kc_log.drop(columns=['id', 'date', 'price', 'lat', 'long'])

X = kc.drop(columns=['id', 'date', 'price', 'yr_renovated',
                                'sqm_above', 'sqm_basement',
                                'sqm_living15', 'sqm_lot15',
                                'lat', 'long'])

y = kc.price

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features (important for Ridge/Lasso)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define models
models = {
"Linear Regression": LinearRegression(),
"Ridge": Ridge(alpha=1.0, random_state=42),
"Lasso": Lasso(alpha=0.5, random_state=42, max_iter=10000),
"ElasticNet": ElasticNet(alpha=0.1, random_state=42)
}

# Function to evaluate models
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"===== {name} =====")
    print(f"R²: {r2:.4f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}", '\n')
    print(pd.Series(dict(zip(X, model.coef_))).abs().sort_values(ascending=False), '\n')

    # Cross-validation (5-fold)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train, y_train, scoring='r2', cv=cv)
    print(f"CV R² mean: {cv_scores.mean():.4f} | std: {cv_scores.std():.4f}")
    print()

# Evaluate all models
for name, model in models.items():
    if name == "Linear Regression":
        evaluate_model(name, model, X_train, X_test, y_train, y_test)
    else:
        evaluate_model(name, model, X_train_scaled, X_test_scaled, y_train, y_test)

===== Linear Regression =====
R²: 0.6522
RMSE: 229315.39
MAE: 143710.48 

waterfront    552280.446446
grade         122510.902811
bathrooms      50015.026364
view           46079.364751
bedrooms       38460.282991
floors         24884.421862
condition      15531.682683
yr_built        3671.470254
sqm_living      1827.919564
sqm_lot            2.995486
dtype: float64 

CV R² mean: 0.6485 | std: 0.0131

===== Ridge =====
R²: 0.6522
RMSE: 229315.69
MAE: 143708.24 

sqm_living    154055.578478
grade         143369.017305
yr_built      108055.927661
waterfront     46600.484558
bathrooms      38353.381659
bedrooms       35809.853565
view           35104.403851
floors         13506.512084
sqm_lot        10805.250705
condition      10123.413888
dtype: float64 

CV R² mean: 0.6485 | std: 0.0131

===== Lasso =====
R²: 0.6522
RMSE: 229315.40
MAE: 143710.34 

sqm_living    154071.399403
grade         143377.038694
yr_built      108067.692488
waterfront     46602.292946
bathrooms      38349.047770
